# milton_da on Colab: train the proxy and the diffusion prior, then run Milton

Before running: **Runtime > Change runtime type > T4 GPU** (or L4 / A100 with Colab Pro).

Expected in `MyDrive/milton_da_upload/`: `milton_da.zip`, `archive_scenes.zip`, `milton_scenes.zip`.
Outputs go to `MyDrive/milton_artifacts/` so they survive a disconnect. Every cell is safe to rerun.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - change the runtime type to GPU')
UP = '/content/drive/MyDrive/milton_da_upload'
OUT = '/content/drive/MyDrive/milton_artifacts'
os.makedirs(OUT, exist_ok=True)
print(os.listdir(UP))

## 1. Unpack code and scenes onto the fast local disk

In [ ]:
%%bash
set -e
UP=/content/drive/MyDrive/milton_da_upload
cd /content
rm -rf milton_da && unzip -q $UP/milton_da.zip -d /content
mkdir -p data/archive data/milton
[ -d data/archive/scenes ] || unzip -q $UP/archive_scenes.zip -d data/archive
[ -d data/milton/scenes ]  || unzip -q $UP/milton_scenes.zip  -d data/milton
echo "archive scenes: $(ls data/archive/scenes | wc -l)   milton scenes: $(ls data/milton/scenes | wc -l)"
pip -q install xarray netCDF4 h5netcdf pyproj 2>&1 | tail -1
python -c "import milton_da, torch; print('import ok, torch', torch.__version__)"

## 2. Train

First run: U-Net proxy (3,000 steps) then the prior (20,000 steps). On a T4 this is roughly 10 min + 45 min.
Checkpoints are written to Drive every 500 prior steps.

In [ ]:
%cd /content
!python -m milton_da.scripts.train --archive data/archive --out /content/drive/MyDrive/milton_artifacts \
    --preset small --downscale 2 --unet-steps 3000 --score-steps 20000 --num-workers 2

### 2b. If the session disconnected during the prior, rerun cells 0 and 1, then this instead of cell 2

In [ ]:
%cd /content
!python -m milton_da.scripts.train --archive data/archive --out /content/drive/MyDrive/milton_artifacts \
    --preset small --downscale 2 --skip-unet --score-steps 20000 --num-workers 2 --resume

## 3. Look at the prior: warm core aloft and ring-like rain, with no observations at all

In [ ]:
from IPython.display import Image, display
display(Image('/content/drive/MyDrive/milton_artifacts/prior_samples.png'))

## 4. Milton retrievals (full system, then the two ablations)

In [ ]:
%cd /content
A='/content/drive/MyDrive/milton_artifacts'
!python -m milton_da.inference.run_milton --scenes data/milton/scenes/MILTON_*.nc --stats $A/norm_stats.json \
    --unet $A/checkpoints/unet.pt --score $A/checkpoints/score.pt --out $A/milton --preset small --downscale 2 --ensemble 8 --steps 500

In [ ]:
%cd /content
A='/content/drive/MyDrive/milton_artifacts'
!python -m milton_da.inference.run_milton --scenes data/milton/scenes/MILTON_*.nc --stats $A/norm_stats.json \
    --unet $A/checkpoints/unet.pt --score $A/checkpoints/score.pt --out $A/milton_no_mw --preset small --downscale 2 --ensemble 8 --steps 500 --no-mw
!python -m milton_da.inference.run_milton --scenes data/milton/scenes/MILTON_*.nc --stats $A/norm_stats.json \
    --unet $A/checkpoints/unet.pt --score $A/checkpoints/score.pt --out $A/milton_no_rtm --preset small --downscale 2 --ensemble 8 --steps 500 --no-rtm

## 5. Quick summary table of the analyses

In [ ]:
import xarray as xr, glob, numpy as np, os
A='/content/drive/MyDrive/milton_artifacts'
for run in ['milton','milton_no_mw','milton_no_rtm']:
    files = sorted(glob.glob(f'{A}/{run}/*_analysis.nc'))
    if not files: continue
    print(f'\n== {run}: {len(files)} scenes')
    for f in files:
        ds = xr.load_dataset(f)
        wc = ds['warm_core_anomaly'].values
        print(os.path.basename(f)[:22], f'warm core max {wc.max():5.1f} K at {int(ds.level[wc.argmax()])} hPa',
              f'T rmse(ERA5) {float(ds["temperature_rmse_vs_era5"].mean()):.2f} K', f'rain rmse(IMERG) {float(ds["precip_rmse_vs_imerg"]):.2f} mm/h')